In [1]:
import sys
sys.path.insert(0, "/home/ci2dt2-ai/Proyectos/Psiquiatria")
import kd_common as kdc
import pandas as pd
import numpy as np
import pickle

cfg = kdc.get_config("ansiedad")
kdc.set_global_seed(kdc.SEED)

master = pd.read_csv(cfg.experiment_dir / "data" / "master_table.csv", dtype={"patient_id": str, "audio_id": str})
split = pd.read_csv(cfg.experiment_dir / "data" / "development_test_split.csv", dtype={"patient_id": str})
folds = pd.read_csv(cfg.experiment_dir / "data" / "folds_5cv.csv", dtype={"patient_id": str})
dev = kdc.build_dev_frame(master, split, folds)
test = kdc.build_test_frame(master, split)
oof = pd.read_csv(cfg.experiment_dir / "teacher" / "teacher_predictions_development_oof.csv", dtype={"patient_id": str})
print(f"dev={len(dev)} test={len(test)} oof={len(oof)}")

dev=4238 test=986 oof=4238


# 02 — Student: barrido LazyClassifier / LazyRegressor (Fases 6-11)

Para cada feature_set (`librosa`, `egemaps`):
- **Base**: `LazyClassifier` sobre y_real (5 folds oficiales) -> mejor clasificador por ROC-AUC medio.
- **KD**: `LazyRegressor` sobre y_KD=0.5*y_real+0.5*p_teacher (5 folds) -> mejor regresor por ROC-AUC medio
  contra y_real (nunca contra el Teacher).
- Ablación completa de alpha (0, .25, .5, .75, 1) con el regresor ganador, reportando los 5 valores
  (no solo el ganador).
- Selección de umbral (Paso 13) para Base y para KD-en-el-mejor-alpha, usando solo OOF de Desarrollo.

In [2]:
results = {}

for feature_set in ["librosa", "egemaps"]:
    print(f"\n{'='*60}\nFEATURE SET: {feature_set}\n{'='*60}")
    X, feat_cols, medians = kdc.load_feature_matrix(cfg, feature_set, dev)
    y = dev["label"].to_numpy().astype(float)
    fold_arr = dev["fold"].to_numpy()

    # Alinear p_teacher_OOF a `dev` por sample_id (nunca asumir mismo orden de filas)
    p_teacher_oof = dev[["sample_id"]].merge(oof[["sample_id", "p_teacher_OOF"]], on="sample_id", how="left")["p_teacher_OOF"].to_numpy()
    assert not np.isnan(p_teacher_oof).any(), "faltan predicciones OOF del Teacher para algunas muestras"

    # --- Base: LazyClassifier sobre y_real ---
    base_scores, base_oof, base_pipes = kdc.lazy_base_cv(X, y.astype(int), fold_arr)
    base_winner, base_summary = kdc.select_best_lazy_model(base_scores, base_oof)
    print(f"Base winner: {base_winner}")
    print(base_summary.head(6).to_string(index=False))

    # --- KD: LazyRegressor sobre y_KD (alpha=0.5 representativo para elegir el modelo) ---
    y_kd_05 = kdc.build_kd_targets(y, p_teacher_oof, 0.5)
    kd_scores, kd_oof, kd_pipes = kdc.lazy_kd_cv(X, y_kd_05, y.astype(int), fold_arr)
    kd_winner, kd_summary = kdc.select_best_lazy_model(kd_scores, kd_oof)
    print(f"KD winner: {kd_winner}")
    print(kd_summary.head(6).to_string(index=False))

    # --- Ablacion completa de alpha con el regresor KD ganador ---
    alpha_df = kdc.lazy_alpha_ablation(X, y, p_teacher_oof, fold_arr, kd_pipes[kd_winner])
    best_alpha = float(alpha_df.loc[alpha_df["roc_auc_mean"].idxmax(), "alpha"])
    print(f"Mejor alpha: {best_alpha}")
    print(alpha_df.to_string(index=False))

    # --- Umbral (Paso 13): sobre OOF de Base y de KD-en-mejor-alpha ---
    base_threshold, base_thr_df = kdc.select_threshold(y.astype(int), base_oof[base_winner])
    y_kd_best = kdc.build_kd_targets(y, p_teacher_oof, best_alpha)
    kd_oof_best_alpha = np.full(len(y), np.nan)
    for k in sorted(int(f) for f in np.unique(fold_arr)):
        tr, va = fold_arr != k, fold_arr == k
        m = kdc.refit_named_model(kd_pipes[kd_winner], X[tr], y_kd_best[tr])
        kd_oof_best_alpha[va] = np.clip(m.predict(X[va]), 0, 1)
    kd_threshold, kd_thr_df = kdc.select_threshold(y.astype(int), kd_oof_best_alpha)
    print(f"Umbral Base: {base_threshold} | Umbral KD: {kd_threshold}")

    out_dir = cfg.experiment_dir / "features" / feature_set
    base_scores.to_csv(out_dir / "base_search_cv.csv", index=False)
    kd_scores.to_csv(out_dir / "kd_search_cv.csv", index=False)
    alpha_df.to_csv(out_dir / "alpha_results_cv.csv", index=False)
    base_thr_df.to_csv(out_dir / "threshold_selection_base.csv", index=False)
    kd_thr_df.to_csv(out_dir / "threshold_selection_kd.csv", index=False)
    with open(out_dir / "base_template_pipe.pkl", "wb") as f:
        pickle.dump(base_pipes[base_winner], f)
    with open(out_dir / "kd_template_pipe.pkl", "wb") as f:
        pickle.dump(kd_pipes[kd_winner], f)
    kdc.save_json(
        {
            "feature_set": feature_set, "base_winner": base_winner, "kd_winner": kd_winner,
            "best_alpha": best_alpha, "base_threshold": base_threshold, "kd_threshold": kd_threshold,
            "base_roc_auc_mean": float(base_summary.iloc[0]["mean"]),
            "kd_roc_auc_mean_at_alpha0.5": float(kd_summary.iloc[0]["mean"]),
        },
        out_dir / "selection.json",
    )
    with open(out_dir / "dev_feature_medians.pkl", "wb") as f:
        pickle.dump(medians, f)
    kdc.save_json({"feature_cols": feat_cols}, out_dir / "feature_columns.json")

    results[feature_set] = {
        "base_winner": base_winner, "kd_winner": kd_winner, "best_alpha": best_alpha,
        "base_threshold": base_threshold, "kd_threshold": kd_threshold,
        "alpha_df": alpha_df, "base_summary": base_summary, "kd_summary": kd_summary,
    }

print("\nListo. selection.json guardado por feature_set en features/<fs>/.")


FEATURE SET: librosa


Base winner: DecisionTreeClassifier
                     model     mean      std
    DecisionTreeClassifier 0.501155 0.042457
        LogisticRegression 0.459273 0.060822
LinearDiscriminantAnalysis 0.455481 0.067263
      KNeighborsClassifier 0.444221 0.052465
         BaggingClassifier 0.438032 0.073498
        AdaBoostClassifier 0.434661 0.060049


KD winner: ElasticNet
                model     mean      std
           ElasticNet 0.500000 0.000000
                Lasso 0.500000 0.000000
DecisionTreeRegressor 0.483527 0.035087
                Ridge 0.480018 0.065005
     LinearRegression 0.479100 0.066012
    AdaBoostRegressor 0.478043 0.134022


Mejor alpha: 0.0
 alpha  roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  f1_mean  f1_std
  0.00           0.5          0.0     0.265737    0.006502      0.0     0.0
  0.25           0.5          0.0     0.265737    0.006502      0.0     0.0
  0.50           0.5          0.0     0.265737    0.006502      0.0     0.0
  0.75           0.5          0.0     0.265737    0.006502      0.0     0.0
  1.00           0.5          0.0     0.265737    0.006502      0.0     0.0
Umbral Base: 0.2 | Umbral KD: 0.2

FEATURE SET: egemaps


Base winner: AdaBoostClassifier
                     model     mean      std
        AdaBoostClassifier 0.498643 0.095400
    DecisionTreeClassifier 0.482118 0.035587
GradientBoostingClassifier 0.481052 0.055810
                GaussianNB 0.475620 0.156689
        LogisticRegression 0.473232 0.101712
LinearDiscriminantAnalysis 0.471783 0.107625


KD winner: AdaBoostRegressor
                    model     mean      std
        AdaBoostRegressor 0.541640 0.150364
         LinearRegression 0.513287 0.097222
                    Ridge 0.512165 0.097533
               ElasticNet 0.500000 0.000000
                    Lasso 0.500000 0.000000
GradientBoostingRegressor 0.495311 0.063494


Mejor alpha: 0.0
 alpha  roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  f1_mean   f1_std
  0.00      0.647073     0.173338     0.419553    0.148095 0.000000 0.000000
  0.25      0.566305     0.147677     0.353486    0.180207 0.000000 0.000000
  0.50      0.541640     0.134490     0.314409    0.103698 0.000000 0.000000
  0.75      0.516940     0.061377     0.270482    0.026118 0.016667 0.033333
  1.00      0.508116     0.055134     0.265461    0.021627 0.132494 0.086919


Umbral Base: 0.33 | Umbral KD: 0.23

Listo. selection.json guardado por feature_set en features/<fs>/.


## Resumen de modelos ganadores

In [3]:
summary_rows = []
for fs, r in results.items():
    summary_rows.append({
        "feature_set": fs, "base_model": r["base_winner"], "kd_model": r["kd_winner"],
        "best_alpha": r["best_alpha"], "base_threshold": r["base_threshold"], "kd_threshold": r["kd_threshold"],
    })
pd.DataFrame(summary_rows)

,feature_set,base_model,kd_model,best_alpha,base_threshold,kd_threshold
0,librosa,DecisionTreeClassifier,ElasticNet,0.0,0.20,0.20
1,egemaps,AdaBoostClassifier,AdaBoostRegressor,0.0,0.33,0.23
